In [ ]:
import math

from steel_lib.si_units import si
from steelpy import aisc

from steel_lib.data_models import (
    Plate,
    ConnectionFactory,
    ConnectionComponent,
    DesignLoads,
  
)
from steel_lib.materials import MATERIALS, BOLT_GRADES, WELD_ELECTRODES
from steel_lib.member_factory import MemberFactory
from steel_lib.calculations import (
    BoltShearCalculator,
    BlockShearCalculator,
    ConnectionCapacityCalculator,
    TensileYieldingCalculator,
    TensileRuptureCalculator,
    TensileYieldWhitmore,
    CompressionBucklingCalculator,
    PlateTensileYieldingCalculator,
    WebLocalYieldingCalculator,
    WebLocalCrippingCalculator,
    ShearYieldingCalculator,
    PryingActionCalculator,
    WeldCalculator,
    ConnectionAnalysis
)

In [2]:
beam = MemberFactory.create_steelpy_member(
    section_class=aisc.W_shapes,
    section_name="W21X83",
    material=MATERIALS["a992"],
    shape_type="W",length=25 * si.ft
)

support = MemberFactory.create_steelpy_member(
    section_class=aisc.W_shapes,
    section_name="W14X90",
    material=MATERIALS["a992"],
    shape_type="W"
)

bracing = MemberFactory.create_steelpy_member(
    section_class=aisc.L_shapes,
    section_name="L8X6X1",
    material=MATERIALS["a36"],
    shape_type="L",
    loading_condition=2,  # Assuming this is a bracing member
)

# End Plate for Column Connection
end_plate_column = Plate(
    t=5/8 * si.inch,
    material=MATERIALS["a572_gr50"],
    width=10 * si.inch,
)
end_plate_beam = Plate(
    t=3/4* si.inch,
    material=MATERIALS["a572_gr50"],
    width=10 * si.inch,
    length = beam.d + 1 * si.inch,  # Assuming the end plate length is equal to the beam depth plus some extra
)
    # Gusset Plate for Bracing Connection
gusset_plate = Plate(
        t=1 * si.inch,
        material=MATERIALS["a572_gr50"],
        clipping=3/4 * si.inch,
    ) 


In [11]:
brace_gusset_loads = DesignLoads(Pu = 840 * si.kip, Vu = 0 * si.kip, Aub = 0 * si.kip)

brace_gusset_connection = ConnectionFactory.create_bolted_connection(
        member_a=bracing,
        member_b=gusset_plate,
        component_a=ConnectionComponent.TOTAL,
        row_spacing=3.0 * si.inch,
        column_spacing=3.0 * si.inch,
        n_rows=2,
        n_columns=7,
        edge_distance_vertical=2 * si.inch,
        edge_distance_horizontal=1.5 * si.inch,
        bolt_diameter=7/8 * si.inch,
        bolt_grade=BOLT_GRADES["a325_x"],
        material=MATERIALS["a572_gr50"],
        angle=47.2 * math.pi / 180
    )
gusset_column_connection = ConnectionFactory.create_bolted_connection(
        member_a=gusset_plate,
        member_b=support,
        component_a=ConnectionComponent.TOTAL,
        row_spacing=3.0 * si.inch,
        column_spacing=3.0 * si.inch,
        n_rows=2,
        n_columns=7,
        edge_distance_vertical=2 * si.inch,
        edge_distance_horizontal=3 * si.inch,
        bolt_diameter=7/8 * si.inch,
        bolt_grade=BOLT_GRADES["a325_x"],
        material=MATERIALS["a572_gr50"],
        angle=47.2 * math.pi / 180
    )
brace_gusset_tensile_yielding = TensileYieldingCalculator(endpoint=brace_gusset_connection.member_a, connection=brace_gusset_connection,load=brace_gusset_loads)
brace_gusset_tensile_yielding.check_dcr(debug=True)
brace_gusset_tensile_rupture = TensileRuptureCalculator(endpoint=brace_gusset_connection.member_a, connection=brace_gusset_connection,load=brace_gusset_loads)
brace_gusset_tensile_rupture.check_dcr(debug=True)
brace_gusset_connection_strength = ConnectionCapacityCalculator(endpoint=brace_gusset_connection.member_b, connection=brace_gusset_connection,loads=brace_gusset_loads)
brace_gusset_connection_strength.check_dcr(debug = True, number_of_shear_planes=2)
brace_gusset_block_shear = BlockShearCalculator(endpoint=brace_gusset_connection.member_a, connection=brace_gusset_connection,loads=brace_gusset_loads, loading_condition=2)
brace_gusset_block_shear.check_dcr(debug=True)


# Gusset Plate Check
gusset_plate_block_shear = BlockShearCalculator(
    endpoint=brace_gusset_connection.member_b,
    connection=brace_gusset_connection,
    loads=brace_gusset_loads,)
gusset_plate_block_shear.check_dcr(debug=True)
gusset_plate_connection_strength = ConnectionCapacityCalculator(
    endpoint=brace_gusset_connection.member_b,
    connection=brace_gusset_connection,
    loads=brace_gusset_loads,
)
gusset_plate_connection_strength.check_dcr(number_of_shear_planes = 2 , debug=True)
gusset_plate_whitmore = TensileYieldWhitmore(
    endpoint=brace_gusset_connection.member_b,
    connection=brace_gusset_connection,loads=brace_gusset_loads)
gusset_plate_whitmore.check_dcr(debug=True)
gusset_plate_compression_buckling = CompressionBucklingCalculator(
    endpoint=brace_gusset_connection.member_b,
    connection=brace_gusset_connection,loads=brace_gusset_loads)
gusset_plate_compression_buckling.calculate_capacity(debug=True)
# gusset_plate_compression_buckling.check_dcr(debug=True)



--- DEBUG: Tensile Yielding (TOTAL) ---
  Inputs:
    Yield Strength (Fy)                : 36.000 ksi
    Applicable Gross Area (Ag) for TOTAL: 13.100 inch²
    Loading Condition                  : 2
    Resistance Factor (phi)            : 0.9000
  Calculations:
    Nominal Strength (Rn = Fy * Ag)    : 471.600 kip
  Output:
                                         --------------------
    Design Strength (phiRn)            : 848.880 kip
--- END DEBUG: Tensile Yielding (TOTAL) ---

--- DEBUG: DCR Check: Tensile Yielding (TOTAL) ---
  Inputs:
    Demand                             : 840.000 kip
    Capacity                           : 848.880 kip
  Output:
                                         --------------------
    DCR (Demand / Capacity)            : 0.9895
--- END DEBUG: DCR Check: Tensile Yielding (TOTAL) ---

--- DEBUG: Tensile Rupture (TOTAL) ---
  Inputs:
    Ultimate Strength (Fu)             : 58.000 ksi
    Applicable Thickness (t)           : 1.000 inch
    Gross Area (

ValueError: Member is not slender enough for compression buckling calculation.

In [ ]:
n_columns= 6
column_spacing = 3.0 * si.inch
beam_column_connection = ConnectionFactory.create_bolted_connection(
        member_a=beam,
        member_b=end_plate_beam,
        component_a=ConnectionComponent.WEB,
        row_spacing=5.5 * si.inch,
        column_spacing=column_spacing,
        n_rows=2,
        n_columns=n_columns,
        edge_distance_vertical=2 * si.inch,
        edge_distance_horizontal=beam.d + 1 * si.inch - (n_columns -1) * column_spacing  - 3 * si.inch,
        bolt_diameter=7/8 * si.inch,
        bolt_grade=BOLT_GRADES["a490_x"],
        material=MATERIALS["a572_gr50"],
    )
initial_loads = DesignLoads(
    Pu=840 * si.kip,
    Vu=50.0 * si.kip,
    Aub=100 * si.kip
)
ufm_calculator = ConnectionAnalysis(beam = beam, support = support, endplate = end_plate_column, brace = bracing,connection=gusset_column_connection,loads=initial_loads,debug=False)
ufm_calculator.run_analysis()
interface_loads = ufm_calculator.interface_loads()
beam_column_interface_loads = interface_loads['Beam_Column_Interface']

bc_bolt_shear = BoltShearCalculator(beam_column_connection,loads=beam_column_interface_loads)
# bc_bolt_shear.calculate_capacity_fnt()
bc_bolt_shear.check_dcr_fnt(debug = False)
bc_prying_action = PryingActionCalculator(end_plate_beam,beam,beam_column_connection,loads = beam_column_interface_loads)
bc_prying_action.check_dcr()
bc_prying_action_column_flange = PryingActionCalculator(support,end_plate_beam,beam_column_connection,loads = beam_column_interface_loads)
bc_prying_action_column_flange.check_dcr()
bc_block_shear = BlockShearCalculator(endpoint=beam_column_connection.member_b, connection=beam_column_connection,loads=beam_column_interface_loads)
bc_block_shear.check_dcr(debug=False)
bc_connection_strength = ShearYieldingCalculator(endpoint=beam_column_connection.member_a, connection=beam_column_connection,loads=beam_column_interface_loads)
bc_connection_strength.check_dcr(debug=True)
bc_connection_strength = ShearYieldingCalculator(endpoint=beam_column_connection.member_a, connection=beam_column_connection,loads=beam_column_interface_loads)
bc_connection_strength.check_dcr(debug=True)


--- DEBUG: Modified Bolt Tensile Strength (Interaction) ---
  Inputs:
    Total Demand Shear Force (V)       : 319.023 kip
    Number of Bolts (n_bolts)          : 12
    Nominal Tensile Stress (Fnt)       : 113.000 ksi
    Nominal Shear Stress (Fnv)         : 84.000 ksi
    Bolt Area (Ab)                     : 0.601 inch²
    Resistance Factor (phi)            : 0.7500
  Calculations:
    Shear per Bolt (V / n_bolts)       : 26.585 kip
    Required Shear Stress (frv = V_per_bolt / Ab): 44.211 ksi
    Interaction Coefficient (Fnt / (phi * Fnv)): 1.7937
    Modified F'nt (Nominal, before cap): 67.600 ksi
  Output:
                                         --------------------
    Final Modified Tensile Stress (F'nt): 30.487 kip
--- END DEBUG: Modified Bolt Tensile Strength (Interaction) ---

--- DEBUG: Modified Bolt Tensile Strength (Interaction) ---
  Inputs:
    Total Demand Shear Force (V)       : 319.023 kip
    Number of Bolts (n_bolts)          : 12
    Nominal Tensile Stress (Fnt

0.964893805418665

In [5]:
BoltShearCalculator()

TypeError: BoltShearCalculator.__init__() missing 1 required positional argument: 'connection'